In [43]:
import numpy as np
import random
from collections import defaultdict

In [45]:
# ---- Step 1: Prepare data ----
corpus = [
    "king queen man woman",
    "paris france berlin germany",
    "cat dog animal",
    "car bus vehicle",
    "apple orange fruit"
]

In [47]:
#  Step 2: Tokenize and build vocabulary
words = [w for sent in corpus for w in sent.split()]
vocab = list(set(words))
word_to_ix = {w: i for i, w in enumerate(vocab)}
ix_to_word = {i: w for w, i in word_to_ix.items()}

VOCAB_SIZE = len(vocab)
EMBED_DIM = 5

In [49]:
# ---- Step 3: Generate training pairs ----
def generate_skipgram(corpus, window_size=1):
    pairs = []
    for sent in corpus:
        tokens = sent.split()
        for i, w in enumerate(tokens):
            target = w
            context_indices = list(range(max(0, i-window_size), min(len(tokens), i+window_size+1)))
            for j in context_indices:
                if j != i:
                    pairs.append((target, tokens[j]))
    return pairs

def generate_cbow(corpus, window_size=1):
    pairs = []
    for sent in corpus:
        tokens = sent.split()
        for i, w in enumerate(tokens):
            context = []
            for j in range(max(0, i-window_size), min(len(tokens), i+window_size+1)):
                if j != i:
                    context.append(tokens[j])
            if context:
                pairs.append((context, w))
    return pairs

sg_data = generate_skipgram(corpus, window_size=1)
cbow_data = generate_cbow(corpus, window_size=1)

print("Skip-Gram sample:", sg_data[:5])
print("CBOW sample:", cbow_data[:5])

Skip-Gram sample: [('king', 'queen'), ('queen', 'king'), ('queen', 'man'), ('man', 'queen'), ('man', 'woman')]
CBOW sample: [(['queen'], 'king'), (['king', 'man'], 'queen'), (['queen', 'woman'], 'man'), (['man'], 'woman'), (['france'], 'paris')]


In [51]:
# 4. Initialize weight matrices
W1_sg = np.random.randn(VOCAB_SIZE, EMBED_DIM)
W2_sg = np.random.randn(EMBED_DIM, VOCAB_SIZE)

W1_cbow = np.random.randn(VOCAB_SIZE, EMBED_DIM)
W2_cbow = np.random.randn(EMBED_DIM, VOCAB_SIZE)

def softmax(x):
    ex = np.exp(x - np.max(x))
    return ex / np.sum(ex)

def one_hot(word):
    vec = np.zeros(VOCAB_SIZE)
    vec[word_to_ix[word]] = 1
    return vec

In [53]:
# 5. Training functions
def train_skipgram(data, epochs=500, lr=0.05):
    global W1_sg, W2_sg
    for epoch in range(epochs):
        loss = 0
        for target, context in data:
            x = one_hot(target)                # input one-hot
            h = np.dot(W1_sg.T, x)             # hidden layer (embedding)
            u = np.dot(W2_sg.T, h)             # output scores
            y_pred = softmax(u)                # probabilities

            y_true = one_hot(context)

            loss += -np.sum(y_true * np.log(y_pred + 1e-9))

            e = y_pred - y_true
            dW2 = np.outer(h, e)
            dW1 = np.outer(x, np.dot(W2_sg, e))

            W1_sg -= lr * dW1
            W2_sg -= lr * dW2

        if (epoch+1) % 100 == 0:
            print("SG Epoch %d, Loss %.4f" % (epoch+1, loss))

def train_cbow(data, epochs=500, lr=0.05):
    global W1_cbow, W2_cbow
    for epoch in range(epochs):
        loss = 0
        for context_list, target in data:
            # input is average of context one-hots
            x = np.zeros(VOCAB_SIZE)
            for w in context_list:
                x += one_hot(w)
            x = x / len(context_list)

            h = np.dot(W1_cbow.T, x)
            u = np.dot(W2_cbow.T, h)
            y_pred = softmax(u)

            y_true = one_hot(target)
            loss += -np.sum(y_true * np.log(y_pred + 1e-9))

            e = y_pred - y_true
            dW2 = np.outer(h, e)
            dW1 = np.outer(x, np.dot(W2_cbow, e))

            W1_cbow -= lr * dW1
            W2_cbow -= lr * dW2

        if (epoch+1) % 100 == 0:
            print("CBOW Epoch %d, Loss %.4f" % (epoch+1, loss))

In [55]:
# 6. Run training
train_skipgram(sg_data, epochs=500, lr=0.05)
train_cbow(cbow_data, epochs=500, lr=0.05)

SG Epoch 100, Loss 12.6310
SG Epoch 200, Loss 11.9574
SG Epoch 300, Loss 11.6444
SG Epoch 400, Loss 11.4251
SG Epoch 500, Loss 11.2635
CBOW Epoch 100, Loss 6.3514
CBOW Epoch 200, Loss 5.3703
CBOW Epoch 300, Loss 5.0943
CBOW Epoch 400, Loss 4.9440
CBOW Epoch 500, Loss 4.8446


In [57]:
# 7. Inspect embeddings
print("\nEmbeddings SG:")
for w in ["king", "queen", "man", "woman"]:
    print(w, W1_sg[word_to_ix[w]])

print("\nEmbeddings CBOW:")
for w in ["king", "queen", "man", "woman"]:
    print(w, W1_cbow[word_to_ix[w]])


Embeddings SG:
king [-1.27872237 -0.30662032 -0.27750619  2.77373329 -0.94508795]
queen [-1.96157413  0.3212342   0.81998905  0.10727226  0.85107307]
man [-0.50316179 -1.55159188 -0.57927907  1.25819552  0.36092555]
woman [-1.46370605  1.47580925  1.49385203 -1.90286909  0.50994998]

Embeddings CBOW:
king [-0.99881332 -2.2619875  -0.85076337  2.47930113 -0.283284  ]
queen [ 1.63360327  0.93956168 -0.36672672 -3.20275808 -0.3560913 ]
man [ 3.16962379 -1.56035589 -0.1614394  -0.04681532  1.02861402]
woman [ 0.5118093  -0.40707039 -1.87335826 -2.33537846  3.66917261]


In [59]:
# ---- Helper to find closest word ----
def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2) + 1e-9)

def most_similar(target_vec, embeddings, top_n=1):
    sims = {}
    for w in vocab:
        sims[w] = cosine_similarity(target_vec, embeddings[word_to_ix[w]])
    sorted_words = sorted(sims.items(), key=lambda item: item[1], reverse=True)
    return sorted_words[:top_n]

In [61]:
print("\n--- Skip-Gram: Predicting context words ---")
target_word = "queen"
target_vec = W1_sg[word_to_ix[target_word]]  # embedding for 'queen'
nearest_contexts = most_similar(target_vec, W1_sg, top_n=5)
print(f"Context words likely near '{target_word}':", nearest_contexts)


--- Skip-Gram: Predicting context words ---
Context words likely near 'queen': [('queen', 0.999999999813404), ('woman', 0.6419390709110396), ('animal', 0.38188603844357666), ('germany', 0.2721753426907122), ('king', 0.22450035949361008)]


In [320]:
print("\n--- Relationship Demo (CBOW) ---")

king_vec = W1_cbow[word_to_ix["king"]]
man_vec = W1_cbow[word_to_ix["man"]]
woman_vec = W1_cbow[word_to_ix["woman"]]

analogy_vec = king_vec - man_vec + woman_vec

nearest = most_similar(analogy_vec, W1_cbow, top_n=3)
print("king - man + woman ≈ ?", nearest)


--- Relationship Demo (CBOW) ---
king - man + woman ≈ ? [('queen', 0.8232159738812621), ('woman', 0.6808681005592878), ('king', 0.5833730120911194)]
